In [1]:
import csv
from enum import Enum
from functools import lru_cache
import heapq
import json
import random
from typing import List, NamedTuple, Tuple
from pathfinding.core.grid import Grid, GridNode
from pathfinding.finder.a_star import AStarFinder
import os

# Set current workdir to file location
# Make sure generative_agents is in your python path, or replace this with a direct absolute path
try:
    from generative_agents.utils import get_project_root
    BASE_PATH = os.path.join(get_project_root(), "assets/matrix/half_ville")
except ImportError:
    # Fallback for Jupyter Notebook execution if utils isn't found
    BASE_PATH = os.path.join(os.getcwd(), "assets/matrix/half_ville")

MazeInfo = NamedTuple("MazeInfo", [("world_name", str), 
                                   ("maze_width", int),
                                   ("maze_height", int),
                                   ("sq_tile_size", int)])

def _load_maze_meta_info():
    """load the json file containing the maze meta information"""
    with open(os.path.join(BASE_PATH, "maze_meta_info.json"), "r") as file:
        data = json.load(file)
    
    return MazeInfo(data["world_name"], data["maze_width"], data["maze_height"], data["sq_tile_size"])

class Level(Enum):
    WORLD = "world"
    SECTOR = "sector"
    ARENA = "arena"
    GAME_OBJECT = "game_object"
    SPAWNING_LOCATION = "spawning_location"

class Tile:
    def __init__(self, x, y, world, sector, arena, game_object, spawning_location, collision, events):
        self.x = x
        self.y = y
        self.world = world
        self.sector = sector
        self.arena = arena
        self.game_object = game_object
        self.spawning_location = spawning_location
        self.collision = collision
        self.events = events
    
    def get_unique_name(self):
        address = ""
        if self.world: 
            address += self.world
        if self.sector:
            address += f":{self.sector}"
        if self.arena:
            address += f":{self.arena}"
        if self.game_object:
            address += f":{self.game_object}"
        return address
    
    def get_path(self, level: Level):
        path = f"{self.world}"

        # FIX: Corrected Enum reference from 'level.WORLD' to 'Level.WORLD'
        if level == Level.WORLD: 
            return path
        else: 
            path += f":{self.sector}"
        
        if level == Level.SECTOR: 
            return path
        else: 
            path += f":{self.arena}"

        if level == Level.ARENA: 
            return path
        else: 
            path += f":{self.game_object}"

        return path
        
    def is_sector(self):
        return self.sector != ""
    
    def is_arena(self):
        return self.arena != ""
    
    def is_game_object(self):
        return self.game_object != ""
    
    def is_spawning_location(self):
        return self.spawning_location != ""
    
    def is_walkable(self):
        return not self.collision
    
    def l2_distance(self, other: 'Tile') -> float:
        return ((self.x - other.x)**2 + (self.y - other.y)**2)**0.5

    def __str__(self):
        return f"Tile(world={self.world}, sector={self.sector}, arena={self.arena}, game_object={self.game_object}, collision={self.collision})"

    def __repr__(self):
        return self.__str__()
    
    def __gt__(self, other: 'Tile') -> bool:
        return (self.x, self.y) > (other.x, other.y)
    
    def __lt__(self, other: 'Tile') -> bool:
        return (self.x, self.y) < (other.x, other.y)
    
    def __eq__(self, other: 'Tile') -> bool:
        return (self.x, self.y) == (other.x, other.y)
    
    def __hash__(self) -> int:
        # FIX: Added parentheses to call get_unique_name()
        return hash((self.get_unique_name(), self.x, self.y))

class SimplePathFinder():
    def __init__(self, grid: List[List[Tile]]):
        self.grid = grid
    
    def find_path(self, start, end):
        open_set = []
        closed_set = set()
        heapq.heappush(open_set, (0, start))
        came_from = {}
        g_score = {pos: float('inf') for row in self.grid for pos in row}
        g_score[start] = 0
        f_score = {pos: float('inf') for row in self.grid for pos in row}
        f_score[start] = self._heuristic(start, end)

        while open_set:
            current = heapq.heappop(open_set)[1]

            if current == end:
                path = self._reconstruct_path(came_from, current)
                return path

            closed_set.add(current)

            for neighbor in self._get_neighbors(current):
                if neighbor in closed_set:
                    continue

                tentative_g_score = g_score[current] + 1

                if tentative_g_score < g_score[neighbor]:
                    came_from[neighbor] = current
                    g_score[neighbor] = tentative_g_score
                    f_score[neighbor] = g_score[neighbor] + self._heuristic(neighbor, end)
                    if neighbor not in [pos for _, pos in open_set]:
                        heapq.heappush(open_set, (f_score[neighbor], neighbor))

        return []

    @staticmethod
    def _heuristic(start, end):
        # FIX: Corrected typo from start.y to end.y
        return abs(start.x - end.x) + abs(start.y - end.y)

    def _get_neighbors(self, pos):
        neighbors = []
        x = pos.x
        y = pos.y

        if x > 0 and self.grid[y][x - 1].is_walkable():
            neighbors.append(self.grid[y][x - 1])
        if x < len(self.grid[0]) - 1 and self.grid[y][x + 1].is_walkable():
            neighbors.append(self.grid[y][x + 1])
        if y > 0 and self.grid[y - 1][x].is_walkable():
            neighbors.append(self.grid[y - 1][x])
        if y < len(self.grid) - 1 and self.grid[y + 1][x].is_walkable():
            neighbors.append(self.grid[y + 1][x])
        return neighbors

    @staticmethod
    def _reconstruct_path(came_from, current):
        path = [current]
        while current in came_from:
            current = came_from[current]
            path.append(current)
        return path[::-1]


class Maze:
    def __init__(self):
        maze_info = _load_maze_meta_info()
        self.maze_name = maze_info.world_name
        self.maze_width = maze_info.maze_width
        self.maze_height = maze_info.maze_height
        self.tile_size = maze_info.sq_tile_size

        self.finder = AStarFinder()
        self.maze = []

        blocks_folder = os.path.join(BASE_PATH, "special_blocks")

        world_blocks = self.read_special_blocks(blocks_folder + "/world_blocks.csv")
        world_block = world_blocks[0][-1]

        sector_blocks = self.read_special_blocks(blocks_folder + "/sector_blocks.csv")
        sector_blocks_dict = {block[0]: block[-1] for block in sector_blocks}
        
        arena_blocks = self.read_special_blocks(blocks_folder + "/arena_blocks.csv")
        arena_blocks_dict = {block[0]: block[-1] for block in arena_blocks}

        game_object_blocks = self.read_special_blocks(blocks_folder + "/game_object_blocks.csv")
        game_object_blocks_dict = {block[0]: block[-1] for block in game_object_blocks}

        spawning_location_blocks = self.read_special_blocks(blocks_folder + "/spawning_location_blocks.csv")
        spawning_location_blocks_dict = {block[0]: block[-1] for block in spawning_location_blocks}

        maze_folder = os.path.join(BASE_PATH, "maze")

        collision_maze_raw = self.read_special_blocks(maze_folder + "/collision_maze.csv")[0]
        sector_maze_raw = self.read_special_blocks(maze_folder + "/sector_maze.csv")[0]
        arena_maze_raw = self.read_special_blocks(maze_folder + "/arena_maze.csv")[0]
        game_object_maze_raw = self.read_special_blocks(maze_folder + "/game_object_maze.csv")[0]
        spawning_location_maze_raw = self.read_special_blocks(maze_folder + "/spawning_location_maze.csv")[0]

        collision_maze = self.convert_flat_list_to_2d_list(collision_maze_raw, self.maze_width)
        sector_maze = self.convert_flat_list_to_2d_list(sector_maze_raw, self.maze_width)
        arena_maze = self.convert_flat_list_to_2d_list(arena_maze_raw, self.maze_width)
        game_object_maze = self.convert_flat_list_to_2d_list(game_object_maze_raw, self.maze_width)
        spawning_location_maze = self.convert_flat_list_to_2d_list(spawning_location_maze_raw, self.maze_width)

        self.tiles = []
        self.grid = Grid(self.maze_width, self.maze_height)

        for i in range(self.maze_height):
            row = []
            for j in range(self.maze_width):
                sector = sector_blocks_dict[sector_maze[i][j]] if sector_maze[i][j] in sector_blocks_dict else ""
                arena = arena_blocks_dict[arena_maze[i][j]] if arena_maze[i][j] in arena_blocks_dict else ""
                game_object = game_object_blocks_dict[game_object_maze[i][j]] if game_object_maze[i][j] in game_object_blocks_dict else ""
                spawning_location = spawning_location_blocks_dict[spawning_location_maze[i][j]] if spawning_location_maze[i][j] in spawning_location_blocks_dict else ""
                collision = collision_maze[i][j] != "0"
                row += [Tile(j, i, world_block, sector, arena, game_object, spawning_location, collision, dict())]           
                node = self.grid.node(j,i)
                node.walkable = not collision
                node.weight = 0 if collision else 1

            self.tiles += [row]

        self.address_tiles = dict()

        for row in self.tiles:
            for tile in row:
                if tile.collision:
                    continue

                address = tile.get_unique_name()

                if address in self.address_tiles: 
                    self.address_tiles[address].append(tile)
                else: 
                    self.address_tiles[address] = [tile]

        self.finder = SimplePathFinder(self.tiles)

    def filter_address_tiles(self, fuzzy_address: str) -> List[Tile]:
        return {address: tiles for address, tiles in self.address_tiles.items() if fuzzy_address in address}

    def get_random_tile(self, tile=None) -> Tile:
        tiles = self.address_tiles[list(self.address_tiles)[random.randint(0, len(self.address_tiles) - 1)]]
        if tile:
            while True:
                random_tile = tiles[random.randint(0, len(tiles) - 1)]
                if random_tile != tile:
                    return random_tile
        return tiles[random.randint(0, len(tiles) - 1)]
    
    def find_path(self, start: Tile, end: Tile) -> List[Tile]:
        path = self.finder.find_path(start, end)
        tiles = []
        for node in path:
            tiles += [self.get_tile(node.x, node.y)]
        return tiles
    
    @lru_cache(maxsize=1000)
    def _find_path(self, start: Tuple[int, int], end: Tuple[int, int]) -> List[GridNode]:
        start_node = self.grid.node(start[0], start[1])
        end_node = self.grid.node(end[0], end[1])
        return self.finder.find_path(start_node, end_node, self.grid)
        
    @lru_cache(maxsize=1000)
    def get_nearby_tiles(self, tile, vision_radius): 
        nearby_tiles = list()
        for i in range(-vision_radius, vision_radius + 1):
            for j in range(-vision_radius, vision_radius + 1):
                if tile.x + i < 0 or tile.x + i >= self.maze_width or tile.y + j < 0 or tile.y + j >= self.maze_height:
                    continue
                nearby_tile = self.get_tile(tile.x + i, tile.y + j)
                if nearby_tile.is_walkable():
                    nearby_tiles += [nearby_tile]
        return nearby_tiles

    @staticmethod
    def convert_flat_list_to_2d_list(flat_list: List[str], width: int) -> List[List[str]]:
        return [flat_list[i:i + width] for i in range(0, len(flat_list), width)]

    @staticmethod
    def read_special_blocks(file_path: str) -> List[List[str]]:
        with open(file_path) as file_handle:
            data_reader = csv.reader(file_handle, delimiter=",")
            return [[cell.strip() for cell in row] for row in data_reader]

    def get_tile(self, x, y):
        return self.tiles[y][x]


if __name__ == "__main__":
    try:
        maze = Maze()
        print("Maze created successfully!")
    except FileNotFoundError:
        print("Could not locate the matrix assets. Please verify BASE_PATH points to the correct directory containing your CSVs and JSON.")

Maze created successfully!


Maze created successfully!


In [6]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import numpy as np

WALL_VAL = 0
WALKABLE_VAL = 1
INTERACTABLE_VAL = 2
AGENT_VAL = 3

def visualize_simulated_world(maze, agent_pos=None, agent_action=None):
    """
    Renders the maze, and if an agent is present, displays their 
    current action via an emoji and a status HUD.
    """
    grid_array = np.zeros((maze.maze_height, maze.maze_width))
    
    for y in range(maze.maze_height):
        for x in range(maze.maze_width):
            tile = maze.get_tile(x, y)
            if not tile.is_walkable():
                grid_array[y][x] = WALL_VAL
            elif tile.is_game_object():
                grid_array[y][x] = INTERACTABLE_VAL
            else:
                grid_array[y][x] = WALKABLE_VAL

    if agent_pos:
        grid_array[agent_pos.y][agent_pos.x] = AGENT_VAL

    color_map = ListedColormap(['black', 'white', 'cornflowerblue', 'crimson'])

    plt.figure(figsize=(12, 10))
    plt.imshow(grid_array, cmap=color_map, vmin=WALL_VAL, vmax=AGENT_VAL, interpolation='none')
    
    # --- NEW: Action Visualization ---
    if agent_pos and agent_action:
        # 1. Map actions to emojis
        action_emojis = {
            "sleeping": "💤",
            "cooking": "🍳",
            "walking": "🚶",
            "thinking": "💭",
            "reading": "📖",
            "interacting": "⚙️",
            "arrived": "✅"
        }
        
        # Get the matching emoji, default to a question mark if unknown
        emoji = action_emojis.get(agent_action.lower(), "❓")
        
        # 2. Draw the emoji slightly above the agent (y - 1)
        plt.text(agent_pos.x, agent_pos.y - 1.5, emoji, 
                 fontsize=18, ha='center', va='center')
        
        # 3. Draw a HUD box in the top-left corner
        hud_text = f"Agent Status:\n{agent_action.capitalize()}"
        plt.text(0.02, 0.98, hud_text, transform=plt.gca().transAxes, 
                 fontsize=12, verticalalignment='top', 
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray'))

    # Keep the existing legend
    legend_elements = [
        Patch(facecolor='black', label='Wall / Collision'),
        Patch(facecolor='white', edgecolor='lightgray', label='Walkable Space'),
        Patch(facecolor='cornflowerblue', label='Interactable Object'),
        Patch(facecolor='crimson', label='Simulated Agent')
    ]
    plt.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.25, 1))
    
    plt.title(f"Simulated World View: {maze.maze_name}")
    plt.axis('off') 
    plt.tight_layout()
    plt.show()

In [ ]:
import time
from IPython.display import clear_output
import matplotlib.pyplot as plt

def animate_agent_path(maze):
    """
    Finds a path from a random starting point to a game object (like a bed)
    and animates the agent moving along that path in the notebook.
    """
    # 1. Find a valid random starting tile
    start_tile = maze.get_random_tile()
    while not start_tile.is_walkable():
        start_tile = maze.get_random_tile()
        
    # 2. Find a destination (Let's prioritize finding a bed)
    end_tile = None
    for address, tiles in maze.address_tiles.items():
        # Check if the address contains "bed" and the tile is walkable
        if "bed" in address.lower() and tiles[0].is_walkable():
            end_tile = tiles[0]
            break
            
    # Fallback: If no bed is found, just find ANY walkable game object
    if not end_tile:
        for address, tiles in maze.address_tiles.items():
            if tiles[0].is_game_object() and tiles[0].is_walkable():
                end_tile = tiles[0]
                break

    # Safety check
    if not end_tile:
        print("Could not find a walkable game object to navigate to.")
        return

    # 3. Calculate the path using your A* implementation
    path = maze.find_path(start_tile, end_tile)
    
    if not path:
        print(f"No valid path found from {start_tile.get_unique_name()} to {end_tile.get_unique_name()}.")
        return
        
    # 4. Animate the movement loop
    for step_index, tile in enumerate(path):
            clear_output(wait=True) 
            
            # Determine the action state
            if step_index == len(path) - 1:
                current_action = "arrived"
            else:
                current_action = "walking"
                
            # Call visualization with the new parameter
            visualize_simulated_world(maze, agent_pos=tile, agent_action=current_action)
            
            time.sleep(0.2)
            
        # Let's add a final frame simulating an interaction!    
        time.sleep(0.5)
        
    clear_output(wait=True)
    
    # If they arrived at a bed, they sleep. Otherwise, they interact.
    final_action = "sleeping" if "bed" in end_tile.get_unique_name().lower() else "interacting"
visualize_simulated_world(maze, agent_pos=path[-1], agent_action=final_action)

# --- Execute the Animation ---
animate_agent_path(maze)

NameError: name 'path' is not defined